<a href="https://colab.research.google.com/github/filipsajtlava/dspracticum2025-tismaci/blob/homework7/homeworks/hw7/homework07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.3 MB/s eta 0:00:00


In [19]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import train_test_split

In [5]:
train_data_path = "http://raw.githubusercontent.com/filipsajtlava/dspracticum2025-tismaci/refs/heads/master/homeworks/hw7/appartments_train.csv"
test_data_path = "https://raw.githubusercontent.com/filipsajtlava/dspracticum2025-tismaci/refs/heads/master/homeworks/hw7/appartments_test.csv"

In [6]:
try:
    df_train = pd.read_csv(train_data_path)
    df_test = pd.read_csv(test_data_path)
    print("Data byla načtena.")
except Exception as e:
    print(f"Chyba: {e}")

Data byla načtena.


In [41]:
categorical = ['layout', 'construction', 'condition', 'ownership', 'elevator', 'address']
not_using = ['text', 'first_seen', 'last_seen', 'price']
using = [col for col in df_train.columns if col not in not_using]

X_train_full = df_train[using].copy()
y_train_full = df_train['price'].copy()
X_test = df_test[using].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=128
)

for col in categorical:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('N/A').astype('str')
        X_test[col] = X_test[col].fillna('N/A').astype('str')
        X_valid[col] = X_valid[col].fillna('N/A').astype('str')


In [30]:
model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.04,
    depth=7,
    loss_function='MAPE',
    eval_metric='MAPE',
    random_seed=128,
    verbose=300,
    cat_features=categorical
)

In [31]:
model.fit(X_train, y_train, eval_set=(X_valid, y_valid), early_stopping_rounds=500)

0:	learn: 0.2677628	test: 0.2839891	best: 0.2839891 (0)	total: 22.7ms	remaining: 34s
300:	learn: 0.1971235	test: 0.2203581	best: 0.2203581 (300)	total: 6.65s	remaining: 26.5s
600:	learn: 0.1844718	test: 0.2104078	best: 0.2104078 (600)	total: 11.7s	remaining: 17.5s
900:	learn: 0.1779365	test: 0.2061422	best: 0.2061422 (900)	total: 18.3s	remaining: 12.2s
1200:	learn: 0.1734032	test: 0.2033504	best: 0.2033504 (1200)	total: 23.4s	remaining: 5.83s
1499:	learn: 0.1693082	test: 0.2010655	best: 0.2010627 (1496)	total: 30.1s	remaining: 0us

bestTest = 0.2010626879
bestIteration = 1496

Shrink model to first 1497 iterations.


In [32]:
feature_importances = pd.Series(model.get_feature_importance(), index=X_train.columns)
top_features = feature_importances.sort_values(ascending=False).head(15)
print(f'Dulezite promenne: {top_features}.')
flop_features = feature_importances[feature_importances < 0.1]
if not flop_features.empty:
    print(f"Málo důležité proměnné ({len(flop_features)}):")
    print(flop_features)

Dulezite promenne: area                               24.109456
layout                             17.105527
condition                          15.950045
construction                       11.805776
ownership                           8.906003
gps_lon                             6.333926
balcony_area                        2.884653
elevator                            2.608844
gps_lat                             2.566185
poi_leisure_time_nearest            2.383842
poi_restaurant_nearest              0.978073
poi_doctors_nearest                 0.965989
poi_grocery_nearest                 0.776280
total_floors                        0.571398
poi_school_kindergarten_nearest     0.552816
dtype: float64.
Málo důležité proměnné (10):
address                          0.000000
cellar_area                      0.059149
garden_area                      0.000000
parking                          0.070965
poi_doctors_count                0.011845
poi_leisure_time_count           0.000000
poi_schoo

In [52]:
categorical = ['layout', 'construction', 'condition', 'ownership', 'elevator']
not_using = ['text', 'first_seen', 'last_seen', 'price']
flop = flop_features.index
flop = flop.tolist()
using = [col for col in df_train.columns if col not in not_using + flop]

X_train_full = df_train[using].copy()
y_train_full = df_train['price'].copy()
X_test = df_test[using].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=128
)

for col in categorical:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('N/A').astype('str')
        X_test[col] = X_test[col].fillna('N/A').astype('str')
        X_valid[col] = X_valid[col].fillna('N/A').astype('str')


In [53]:
model = CatBoostRegressor(
    iterations=30000,
    learning_rate=0.015,
    depth=8,
    loss_function='MAPE',
    eval_metric='MAPE',
    random_seed=128,
    verbose=500,
    cat_features=categorical
)

In [54]:
model.fit(X_train, y_train, eval_set=(X_valid, y_valid), early_stopping_rounds=300)

0:	learn: 0.2691700	test: 0.2853762	best: 0.2853762 (0)	total: 31.2ms	remaining: 15m 36s
500:	learn: 0.1908618	test: 0.2165110	best: 0.2165110 (500)	total: 9.53s	remaining: 9m 21s
1000:	learn: 0.1718707	test: 0.2026069	best: 0.2026069 (1000)	total: 25.3s	remaining: 12m 13s
1500:	learn: 0.1620403	test: 0.1957978	best: 0.1957978 (1500)	total: 36.6s	remaining: 11m 35s
2000:	learn: 0.1549705	test: 0.1913449	best: 0.1913449 (2000)	total: 49.2s	remaining: 11m 28s
2500:	learn: 0.1489218	test: 0.1878704	best: 0.1878704 (2500)	total: 1m 1s	remaining: 11m 15s
3000:	learn: 0.1444458	test: 0.1850354	best: 0.1850354 (3000)	total: 1m 14s	remaining: 11m 5s
3500:	learn: 0.1407505	test: 0.1830569	best: 0.1830569 (3500)	total: 1m 26s	remaining: 10m 53s
4000:	learn: 0.1374355	test: 0.1811501	best: 0.1811501 (4000)	total: 1m 38s	remaining: 10m 40s
4500:	learn: 0.1343075	test: 0.1796212	best: 0.1796212 (4500)	total: 1m 52s	remaining: 10m 38s
5000:	learn: 0.1315915	test: 0.1782505	best: 0.1782505 (5000)	tot

In [56]:
y_pred_test = model.predict(X_test)

In [57]:
prediction = pd.DataFrame({
    'id': df_test['id'],
    'price': y_pred_test.round(0).astype(int)
})
prediction.to_csv('predikce.csv')